In [ ]:
# colab 設定
# 学習時はcontent配下にDATA_ROOTを置く。
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
import os
import shutil
from pathlib import Path
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets
from torchvision.transforms import v2
DRIVE_DATA_ROOT = "/content/gdrive/MyDrive/深層学習スクラッチ/deeplearning_implementation/AlexNet/data/CIFAR10"
DATA_ROOT = "/content/data/CIFAR10"
DRIVE_SETTING_ROOT = "/content/gdrive/MyDrive/深層学習スクラッチ/deeplearning_implementation/AlexNet/settings"

In [ ]:
dataset = datasets.CIFAR10(root=DRIVE_DATA_ROOT, train=True, download=True)
shutil.copytree(DRIVE_DATA_ROOT, DATA_ROOT, dirs_exist_ok=True)

In [ ]:
def compute_rgb_statistics(loader):
    rgb_sum = torch.zeros(3, dtype=torch.float64)
    rgb_outer = torch.zeros((3, 3), dtype=torch.float64)
    n_pixels = 0

    for images, _ in loader:
        # images: [B, 3, H, W]
        pixels = images.permute(0, 2, 3, 1).reshape(-1, 3).double()
        rgb_sum += pixels.sum(dim=0)
        rgb_outer += pixels.T @ pixels
        n_pixels += pixels.shape[0]
    mean = rgb_sum / n_pixels
    covariance = rgb_outer / n_pixels - torch.outer(mean, mean)
    return mean.float(), covariance.float()

class MeanSubtraction:
    def __init__(self, mean):
        self.mean = torch.as_tensor(mean, dtype=torch.float32).view(3, 1, 1)

    def __call__(self, image):
        return image - self.mean.to(image.device)

class PCAColorAugmentation:
    def __init__(self, eigenvectors, eigenvalues, alpha_std=0.1):
        self.eigenvectors = torch.as_tensor(eigenvectors, dtype=torch.float32)
        self.eigenvalues = torch.as_tensor(eigenvalues, dtype=torch.float32)
        self.alpha_std = alpha_std

    def __call__(self, image):
        alpha = torch.randn(3, dtype=image.dtype, device=image.device) * self.alpha_std
        self.eigenvectors = self.eigenvectors.to(image.device)
        self.eigenvalues = self.eigenvalues.to(image.device)
        rgb_shift = self.eigenvectors @ (alpha * self.eigenvalues)
        rgb_shift = rgb_shift.view(3, 1, 1 )
        return image + rgb_shift




In [ ]:
class CIFAR10Dataset(Dataset):
    def __init__(self, root, split, transform=None, download=True):
        self.root = Path(root)
        self.split = split
        self.transform = transform

        #split
        if split == "train":
            train = True
        elif split == "test":
            train = False
        else:
            raise ValueError(f'split must be "train" or "test".')

        self.dataset = datasets.CIFAR10(root=self.root, train=train, download=download)
        self.class_name = self.dataset.classes
        self.class_to_idx = self.dataset.class_to_idx

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        image, label = self.dataset[index]
        if self.transform is not None:
            image = self.transform(image)
        return image, label

    def get_class_name(self, class_idx):
        return self.class_name[class_idx]

In [ ]:
%mkdir $DRIVE_SETTING_ROOT
STATS_PATH = Path(DRIVE_SETTING_ROOT) / "stats.pth"

if STATS_PATH.exists():
    print(f"load statistics: {STATS_PATH}")
    stats = torch.load(STATS_PATH, map_location="cpu")
    mean = stats["mean"]
    covariance = stats["covariance"]
    eigenvalues = stats["eigenvalues"]
    eigenvectors = stats["eigenvectors"]

else:
    print("computing statics")
    stats_transform = v2.Compose(
        [
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True)
        ]
    )
    stats_dataset = CIFAR10Dataset(root=DATA_ROOT, split="train", transform=stats_transform)
    stats_loader = DataLoader(
        stats_dataset,
        batch_size=128,
        shuffle=False,
        num_workers=0
    )
    mean, covariance = compute_rgb_statistics(stats_loader)
    eigenvalues, eigenvectors = torch.linalg.eigh(covariance)

    STATS_PATH.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"mean": mean, "covariance": covariance, "eigenvalues": eigenvalues, "eigenvectors": eigenvectors}, STATS_PATH)
    print(f"save statistics: {STATS_PATH}")


print("mean:")
print(mean)
print("covariance:")
print(covariance)
print("eigenvalues:")
print(eigenvalues)
print("eigenvectors:")
print(eigenvectors)
mean_subtraction = MeanSubtraction(mean)
pca_augmentation = PCAColorAugmentation(eigenvectors=eigenvectors, eigenvalues=eigenvalues, alpha_std=0.1)


In [ ]:
train_transform = v2.Compose(
    [
        v2.ToImage(),
        #v2.Resize(256),
        #v2.RandomCrop(224),
        v2.RandomHorizontalFlip(),
        v2.ToDtype(torch.float32, scale=True),
        pca_augmentation,
        mean_subtraction
    ]
)

val_transform = v2.Compose(
    [
        v2.ToImage(),
        #v2.Resize(256),
        v2.ToDtype(torch.float32, scale=True),
        mean_subtraction
    ]
)

debug_transform = v2.Compose(
    [
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        mean_subtraction
    ]
)

debug_base_dataset = CIFAR10Dataset(
    root=DATA_ROOT,
    split="train",
    transform=debug_transform
)

In [ ]:
train_base_dataset = CIFAR10Dataset(root=DATA_ROOT, split="train", transform=train_transform)
val_base_dataset = CIFAR10Dataset(root=DATA_ROOT, split="train", transform=val_transform)
test_dataset = CIFAR10Dataset(root=DATA_ROOT,split="test",transform=val_transform)
debug_base_dataset = CIFAR10Dataset(root=DATA_ROOT, split="train", transform=debug_transform)

num_total = len(train_base_dataset)
num_val = 5000
num_train = num_total - num_val
generator = torch.Generator().manual_seed(42)
indices = torch.randperm(num_total, generator=generator).tolist()
train_indices = indices[:num_train]
val_indices = indices[num_train:]
train_dataset = Subset(train_base_dataset, train_indices)
val_dataset = Subset(val_base_dataset, val_indices)
debug_dataset = Subset(debug_base_dataset, range(100))


print("train:", len(train_dataset))
print("val:", len(val_dataset))
print("test:", len(test_dataset))

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=128, # 要確認
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

debug_loader = DataLoader(
    debug_dataset,
    batch_size=25,
    shuffle=True,
    num_workers=0
)
DEBUG = False
if DEBUG:
    train_loader = debug_loader
    val_loader = debug_loader
    test_loader = debug_loader

In [ ]:
images, labels = next(iter(train_loader))
print(images.shape)
print(labels.shape)
print(labels[:10])

In [ ]:
class AlexNetLRN(nn.Module):
    def __init__(self, size=5, alpha=1e-4, beta=0.75, k=2.0):
        super().__init__()
        self.size = size
        self.alpha = alpha
        self.beta = beta
        self.k = k

    def forward(self, x):
        # x: [B, C, H, W]
        squared = x.pow(2)
        pad = self.size // 2
        squared = F.pad(squared, (0, 0, 0, 0, pad, pad))
        scale = torch.zeros_like(x)
        for i in range(self.size):
            scale += squared[:, i:i+x.size(1), :, :]
        scale = self.k + (self.alpha / self.size) * scale
        return x / scale.pow(self.beta)

In [ ]:
class AlexNet32(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # Conv1
            nn.Conv2d(in_channels=3, out_channels=96, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            #nn.Identity(),
            nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Conv2
            nn.Conv2d(in_channels=96, out_channels=256, kernel_size=5, stride=1, padding=2, groups=2),
            nn.ReLU(inplace=True),
            #nn.Identity(),
            nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Conv3
            nn.Conv2d(in_channels=256, out_channels=384, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            # Conv4
            nn.Conv2d(in_channels=384, out_channels=384, kernel_size=3, stride=1, padding=1, groups=2),
            nn.ReLU(inplace=True),
            # Conv5
            nn.Conv2d(in_channels=384, out_channels=256, kernel_size=3, stride=1, padding=1, groups=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            # FC6
            nn.Dropout(p=0.5),
            nn.Linear(256 * 4 * 4, 1024),
            nn.ReLU(inplace=True),
            # FC7
            nn.Dropout(p=0.5),
            nn.Linear(1024, 1024),
            nn.ReLU(inplace=True),
            # FC8
            nn.Linear(1024, num_classes)
        )
        self._initialize_weights()

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

    # def _initialize_weights(self):
    #     for module in self.modules():
    #         if isinstance(module, nn.Conv2d):
    #             nn.init.normal_(module.weight, mean=0.0, std=0.01)
    #             nn.init.constant_(module.bias, 0.0)
    #         elif isinstance(module, nn.Linear):
    #             nn.init.normal_(module.weight, mean=0.0, std=0.01)
    #             nn.init.constant_(module.bias, 0.0)
    #     nn.init.constant_(self.features[4].bias, 1.0)
    #     nn.init.constant_(self.features[10].bias, 1.0)
    #     nn.init.constant_(self.features[12].bias, 1.0)
    #     nn.init.constant_(self.classifier[1].bias, 1.0)
    #     nn.init.constant_(self.classifier[4].bias, 1.0)

    def _initialize_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(
                    module.weight,
                    mode="fan_out",
                    nonlinearity="relu"
                )
                if module.bias is not None:
                    nn.init.constant_(
                        module.bias,
                        0.0
                    )
            elif isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(
                    module.weight,
                    mode="fan_in",
                    nonlinearity="relu"
                )
                if module.bias is not None:
                    nn.init.constant_(
                        module.bias,
                        0.0
                    )

In [ ]:
model = AlexNet32(num_classes=10)
print(
    "conv1:",
    model.features[0].bias[0]
)

print(
    "conv2:",
    model.features[4].bias[0]
)

print(
    "conv3:",
    model.features[8].bias[0]
)

print(
    "conv4:",
    model.features[10].bias[0]
)

print(
    "conv5:",
    model.features[12].bias[0]
)

print(
    "fc6:",
    model.classifier[1].bias[0]
)

print(
    "fc7:",
    model.classifier[4].bias[0]
)

print(
    "fc8:",
    model.classifier[6].bias[0]
)

In [ ]:
def print_weight_stats(model):
    print("---- Weight stats ----")
    for name, param in model.named_parameters():
        if "weight" in name:
            print(
                f"{name:30s} "
                f"abs_mean={param.data.abs().mean().item():.8e} "
                f"std={param.data.std().item():.8e} "
                f"abs_max={param.data.abs().max().item():.8e}"
            )
    print("----------------------")

In [ ]:
checkpoint_dir = Path(DRIVE_SETTING_ROOT)
checkpoint_dir.mkdir(parents=True, exist_ok=True)
latest_path = checkpoint_dir / "latest_path.pth"
best_path = checkpoint_dir / "best_path.pth"

# m1 mac用
# device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
# cuda製GPU用
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = model.to(device)
criterion= nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.1, patience=3)


if latest_path.exists():
    print(f"load checkpoint: {latest_path}")
    checkpoint = torch.load(latest_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    best_val_acc = checkpoint.get("best_val_acc", 0.0)
    train_loss_history = checkpoint.get("train_loss_history", [])
    train_acc_history = checkpoint.get("train_acc_history", [])
    val_loss_history = checkpoint.get("val_loss_history", [])
    val_acc_history = checkpoint.get("val_acc_history", [])
    print(f"Resume from epoch {start_epoch}")
    print(f"Best Val Acc " f"{best_val_acc:.4f}")
else:
    start_epoch = 0
    best_val_acc = 0.0
    train_loss_history = []
    train_acc_history = []
    val_loss_history = []
    val_acc_history = []
    print("No checkpoint found. " "Start training from scratch.")

print("start_epoch:", start_epoch)
print("learning rate:", optimizer.param_groups[0]["lr"])

num_epochs = 90
for epoch in range(start_epoch, num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(dim=1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    train_loss = running_loss / total
    train_acc = correct / total
    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss_sum += loss.item() * images.size(0)
            predicted = outputs.argmax(dim=1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total
    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch [{epoch + 1}/ {num_epochs}] " f"Train Loss: {train_loss:.4f} " f"Train Acc: {train_acc:.4f} " f"Val Loss: {val_loss:.4f} " f"Val Acc: {val_acc:.4f} " f"LR: {current_lr:.6f}")
    train_loss_history.append(train_loss)
    train_acc_history.append(train_acc)
    val_loss_history.append(val_loss)
    val_acc_history.append(val_acc)
    #scheduler.step(val_loss)
    print_weight_stats(model)
    # 履歴保存
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_val_acc": best_val_acc,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "train_loss_history": train_loss_history,
            "train_acc_history": train_acc_history,
            "val_loss_history": val_loss_history,
            "val_acc_history": val_acc_history
        }, best_path)
        print(f"Best model saved." f"{best_path}")

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_acc": best_val_acc,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "train_loss_history": train_loss_history,
        "train_acc_history": train_acc_history,
        "val_loss_history": val_loss_history,
        "val_acc_history": val_acc_history
    }, latest_path)
    print(f"Latest checkpoint saved." f"{latest_path}")






In [ ]:
import matplotlib.pyplot as plt
epochs = range(1, len(train_loss_history) + 1)

# Accuracyを%に変換
train_acc_percent = [acc * 100 for acc in train_acc_history]

val_acc_percent = [acc * 100 for acc in val_acc_history]

# グラフ作成
fig, ax1 = plt.subplots(figsize=(12, 7))

# 左Y軸: Loss
line1 = ax1.plot(epochs, train_loss_history, label="Train Loss")
line2 = ax1.plot(epochs, val_loss_history, label="Validation Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.grid()

# 右Y軸: Accuracy
ax2 = ax1.twinx()
line3 = ax2.plot(epochs, train_acc_percent, label="Train Accuracy")
line4 = ax2.plot(epochs, val_acc_percent, label="Validation Accuracy")
ax2.set_ylabel("Accuracy (%)")
ax2.set_ylim(0, 100)

lines = line1 + line2 + line3 + line4
labels = [line.get_label() for line in lines]
ax1.legend(lines, labels, loc="center right")
plt.title("Training and Validation Loss / Accuracy")
plt.tight_layout()
plt.show()

In [ ]:
from collections import Counter

model.eval()
all_labels = []
all_preds = []

with torch.no_grad():
    for images, labels in train_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu()
        all_labels.extend(labels.tolist())
        all_preds.extend(preds.tolist())
print("True labels:")
print(Counter(all_labels))
print("\nPredictions:")
print(Counter(all_preds))

In [ ]:
model = AlexNet32(num_classes=10).to(device)
checkpoint = torch.load(best_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
# Test
test_loss_sum = 0.0
test_correct = 0
test_total = 0
all_images = []
all_labels = []
all_predictions = []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss_sum += (loss.item() * images.size(0))
        predicted = outputs.argmax(dim=1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()
        all_images.append(images.cpu())
        all_labels.append(labels.cpu())
        all_predictions.append(predicted.cpu())
test_loss = (test_loss_sum / test_total)
test_acc = (test_correct / test_total)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: " f"{test_acc:.4f} " f"({test_acc * 100:.2f}%)")